# NLP Assignment 5: Token Classification using BERT

### Objective:
To build and fine-tune a BERT model for token classification tasks such as POS tagging and chunking using Hugging Face Transformers.

 # Import Libraries

In [35]:
## 1. Importing Required Libraries

In [1]:
!pip install seqeval --use-pep517

In [2]:
!pip install --upgrade datasets

In [3]:
!pip install transformers datasets seqeval

# Dataset Loading

In [36]:
# In this step, we load the CoNLL-2003 dataset manually for Named Entity Recognition (NER).

In [4]:
def read_conll(file_path):
    sentences = []
    labels = []

    with open(file_path, "r", encoding="utf-8") as f:
        words = []
        tags = []

        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words = []
                    tags = []
            else:
                splits = line.strip().split()
                words.append(splits[0])
                tags.append(splits[-1])

    return sentences, labels

In [5]:
train_sentences, train_labels = read_conll("C:\\Users\\USER VICTUS\\Downloads\\archive\\train.txt")

 # Data Cleaning

In [37]:
print(train_sentences[0])
print(train_labels[0])
# Remove unnecessary tokens like '-DOCSTART-' from the dataset.
clean_sentences = []
clean_labels = []

for sent, lab in zip(train_sentences, train_labels):
    if sent[0] != "-DOCSTART-":
        clean_sentences.append(sent)
        clean_labels.append(lab)

['-DOCSTART-']
['O']


In [8]:
print(clean_sentences[0])
print(clean_labels[0])

['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


# Label Mapping

In [38]:
# Convert text labels into numerical format for model training.

In [9]:
label_list = list(set(label for sent in clean_labels for label in sent))

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)

{'B-MISC': 0, 'B-ORG': 1, 'B-PER': 2, 'I-ORG': 3, 'I-PER': 4, 'O': 5, 'I-LOC': 6, 'I-MISC': 7, 'B-LOC': 8}


In [10]:
def convert_labels_to_ids(labels):
    return [[label2id[label] for label in sent] for sent in labels]

train_labels_ids = convert_labels_to_ids(clean_labels)

In [11]:
print(train_labels_ids[0])

[1, 5, 0, 5, 5, 5, 0, 5, 5]


# Tokenization + Alignment

In [39]:
# Tokenize sentences using BERT tokenizer and align labels with tokens.

In [12]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [13]:
def tokenize_and_align(sentences, labels):
    tokenized_inputs = tokenizer(
        sentences,
        is_split_into_words=True,
        padding=True,
        truncation=True
    )

    aligned_labels = []

    for i, label in enumerate(labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # special tokens
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)  # subwords
            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [14]:
small_sentences = clean_sentences[:1000]
small_labels = train_labels_ids[:1000]

tokenized_data = tokenize_and_align(small_sentences, small_labels)

In [15]:
print(tokenized_data.keys())

KeysView({'input_ids': [[101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 1943, 14428, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 26660, 13329, 12649, 15928, 1820, 118, 4775, 118, 1659, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 1109, 1735, 2827, 1163, 1113, 9170, 1122, 19786, 1114, 1528, 5566, 1106, 11060, 1106, 188, 17315, 1418, 2495, 12913, 1235, 6479, 4959, 24

# Prepare Dataset

In [40]:
# Convert tokenized data into Hugging Face Dataset format.

In [21]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "input_ids": tokenized_data["input_ids"],
    "attention_mask": tokenized_data["attention_mask"],
    "labels": tokenized_data["labels"]
})

# Model Setup

In [41]:
# Load pre-trained BERT model for token classification.

In [22]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

# Training

In [42]:
# Train the model using Hugging Face Trainer API.

In [44]:
from transformers import TrainingArguments
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    logging_steps=10
)

In [43]:
from transformers import Trainer
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [29]:
trainer.train()

C:\Users\USER VICTUS\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,1.509527
20,0.861007
30,0.741151
40,0.637233
50,0.571515
60,0.365418
70,0.396616
80,0.329251
90,0.311573
100,0.264336


TrainOutput(global_step=125, training_loss=0.5297239227294922, metrics={'train_runtime': 576.1891, 'train_samples_per_second': 1.736, 'train_steps_per_second': 0.217, 'total_flos': 42361334658000.0, 'train_loss': 0.5297239227294922, 'epoch': 1.0})

# Evaluation

In [30]:
predictions = trainer.predict(train_dataset)

C:\Users\USER VICTUS\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [31]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=2)
labels = predictions.label_ids

In [32]:
true_labels = []
true_preds = []

for pred, lab in zip(preds, labels):
    temp_preds = []
    temp_labels = []
    
    for p, l in zip(pred, lab):
        if l != -100:
            temp_preds.append(id2label[p])
            temp_labels.append(id2label[l])
    
    true_preds.append(temp_preds)
    true_labels.append(temp_labels)

In [33]:
from seqeval.metrics import classification_report

print(classification_report(true_labels, true_preds))

              precision    recall  f1-score   support

         LOC       0.60      0.81      0.69       694
        MISC       0.45      0.02      0.04       237
         ORG       0.35      0.47      0.40       379
         PER       0.95      0.95      0.95       631

   micro avg       0.65      0.69      0.67      1941
   macro avg       0.59      0.56      0.52      1941
weighted avg       0.65      0.69      0.64      1941



In [34]:
sentence = "John works at Google in California"

words = sentence.split()

inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt")

outputs = model(**inputs).logits
predictions = outputs.argmax(dim=-1)

predicted_labels = []

for i, word_id in enumerate(inputs.word_ids()):
    if word_id is not None:
        predicted_labels.append(id2label[predictions[0][i].item()])

print(list(zip(words, predicted_labels)))

[('John', 'B-PER'), ('works', 'O'), ('at', 'O'), ('Google', 'O'), ('in', 'O'), ('California', 'B-LOC')]


#  Report

Comparison

POS Tagging vs Chunking

POS Tagging:

Word-level tagging hota hai
Example: noun, verb, adjective
Easy task
Grammar samajhne ke liye use hota hai

Chunking (NER in your case):

Phrase / entity-level detection
Example: Person, Location, Organization
Thoda complex
Context samajhna padta hai

Challenges Faced

Dataset loading issues (version conflicts)

Label alignment with subwords

Handling -100 values

Training on CPU (slow)


Observations

Model performed best on PER entities

MISC category had poor performance

Increasing training data improves accuracy

BERT captures contextual meaning effectively








































##  Conclusion

The BERT model performed well on entity recognition, especially for PERSON entities. Performance can be improved with more data and training epochs.